# Model Retraining — Encrypted Malicious Traffic (NFStream features)

Retrains the detection models on `data/training_dataset.csv` (real Windows-malware
C2 + exfil over TLS, balanced ~11.8k flows) and re-serialises `models/mapper` for the
inference service.

**Pipeline:** load → scale + split → train RF / XGBoost / EBM → rank features →
`rf_best` on top features → save `mapper`.

**Features:** the 29 NFStream-computable features (the 4 TTL features are excluded —
NFStream 6.6.0 exposes no TTL). Run the cells top-to-bottom.


## 0. Imports

In [ ]:
import os
import warnings
warnings.simplefilter("ignore")

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier   # EBM


## 1. Load dataset & build feature matrix

In [ ]:
# Locate the dataset (notebook lives in model_training/)
DATA = next((p for p in ["../data/training_dataset.csv", "data/training_dataset.csv",
                          os.path.expanduser("~/dev/data/training_dataset.csv")]
             if os.path.exists(p)), None)
assert DATA, "training_dataset.csv not found - run utils/build_training_dataset.py first"
df = pd.read_csv(DATA)

# Non-feature columns + dead TTL features (NFStream 6.6.0 exposes no TTL -> all 0)
META = ["true_label", "class", "attack_type", "source", "flow_id", "src_ip", "dst_ip",
        "src_port", "dst_port", "protocol", "bidirectional_packets", "requested_server_name"]
DEAD_TTL = ["mean_time_to_live", "std_time_to_live", "max_time_to_live", "min_time_to_live"]

NFSTREAM_FEATURES = [c for c in df.columns if c not in META + DEAD_TTL]
REALTIME_SAFE_FEATURES = NFSTREAM_FEATURES        # all NFStream features are real-time safe
target_cols = ["label"]

main_df = df[NFSTREAM_FEATURES].copy()
main_df["label"] = df["true_label"].astype(int)

print(f"flows   : {len(main_df):,}")
print(f"features: {len(NFSTREAM_FEATURES)}")
print(f"balance : {main_df['label'].value_counts().to_dict()}")


## 2. Scale (MinMax) + stratified 80/20 split
Scalers are fit on the **raw** features and saved in the mapper; the inference service applies them before predicting, so models are trained on scaled data.

In [ ]:
X_raw = (main_df[REALTIME_SAFE_FEATURES]
         .apply(pd.to_numeric, errors="coerce")
         .replace([np.inf, -np.inf], np.nan))
X_raw = X_raw.fillna(X_raw.median())

scaler_rt = MinMaxScaler().fit(X_raw)
X = pd.DataFrame(scaler_rt.transform(X_raw), columns=REALTIME_SAFE_FEATURES, index=X_raw.index)
y = main_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("train:", X_train.shape, " test:", X_test.shape)
print("train balance:", y_train.value_counts().to_dict())


## 3. Evaluation helper

In [ ]:
def evaluate(model, X_te, y_te, name=""):
    """Accuracy / F1 / ROC-AUC + report on the held-out test set."""
    y_pred = model.predict(X_te)
    y_score = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None
    acc = accuracy_score(y_te, y_pred)
    f1m = f1_score(y_te, y_pred, average="macro")
    auc = roc_auc_score(y_te, y_score) if y_score is not None else float("nan")
    print(f"=== {name} ===")
    print(f"Accuracy {acc:.4f} | F1(macro) {f1m:.4f} | ROC-AUC {auc:.4f}")
    print(classification_report(y_te, y_pred, target_names=["benign", "malicious"], zero_division=0))
    print("Confusion matrix [tn fp / fn tp]:\n", confusion_matrix(y_te, y_pred))
    return {"accuracy": acc, "f1_macro": f1m, "roc_auc": auc}


## 4. Train models on all 29 features
RF, XGBoost (`xgb_rt`), and EBM (`xxgb`) — the three models the inference service loads. `class_weight`/`scale_pos_weight` are left at defaults since the train set is balanced 1:1; enable them if you rebuild the dataset without `--balance`.

In [ ]:
rf_rt = RandomForestClassifier(
    n_estimators=600, max_features="sqrt", min_samples_split=2, min_samples_leaf=1,
    n_jobs=-1, random_state=42)
rf_rt.fit(X_train, y_train)
_ = evaluate(rf_rt, X_test, y_test, "RandomForest (rf_rt)")


In [ ]:
xgb_rt = XGBClassifier(
    n_estimators=600, max_depth=9, learning_rate=0.1, subsample=0.7,
    colsample_bytree=1.0, min_child_weight=1, gamma=0.1, reg_lambda=0.5, reg_alpha=0.001,
    eval_metric="logloss", n_jobs=-1, random_state=42)
xgb_rt.fit(X_train, y_train)
_ = evaluate(xgb_rt, X_test, y_test, "XGBoost (xgb_rt)")


In [ ]:
# EBM = glass-box model used for Tier-1 always-on explanations
xxgb = ExplainableBoostingClassifier(
    learning_rate=0.05, outer_bags=14, inner_bags=0, max_leaves=3,
    min_samples_leaf=2, interactions=10, max_bins=1024, random_state=42)
xxgb.fit(X_train, y_train)
_ = evaluate(xxgb, X_test, y_test, "EBM (xxgb)")


## 5. Feature importance → `BEST_FEATURES`
Rank by averaged RF + XGBoost importance and keep the top-k (default 9) as the compact `BEST_FEATURES` set used by `rf_best`.

In [ ]:
TOP_K = 9
imp = (pd.Series(rf_rt.feature_importances_, index=REALTIME_SAFE_FEATURES).rank(ascending=False)
       + pd.Series(xgb_rt.feature_importances_, index=REALTIME_SAFE_FEATURES).rank(ascending=False))
ranking = imp.sort_values().index.tolist()        # lower combined rank = more important
best_features = ranking[:TOP_K]

print(f"Top {TOP_K} BEST_FEATURES:")
for i, f in enumerate(best_features, 1):
    print(f"  {i:2d}. {f}")


In [ ]:
rf_best = RandomForestClassifier(
    n_estimators=600, max_features="sqrt", min_samples_split=2, min_samples_leaf=1,
    n_jobs=-1, random_state=42)
rf_best.fit(X_train[best_features], y_train)
_ = evaluate(rf_best, X_test[best_features], y_test, f"RandomForest on BEST_FEATURES (rf_best)")


## 6. Save `mapper`
Re-serialises the same dict structure the inference service expects, to `models/mapper`.

In [ ]:
# Scalers fit on RAW features (inference applies them before predicting)
scaler_rt   = MinMaxScaler().fit(main_df[REALTIME_SAFE_FEATURES])
scaler_best = MinMaxScaler().fit(main_df[best_features])

mapper = {
    "rf_best": rf_best,
    "xgb_rt": xgb_rt,
    "xxgb": xxgb,
    "scaler_rt": scaler_rt,
    "scaler_best": scaler_best,
    "REALTIME_SAFE_FEATURES": list(REALTIME_SAFE_FEATURES),
    "BEST_FEATURES": list(best_features),
    "target_cols": target_cols,
}

out = next((d for d in ["../models", "models", os.path.expanduser("~/dev/models")]
            if os.path.isdir(d)), "../models")
os.makedirs(out, exist_ok=True)
path = os.path.join(out, "mapper")
joblib.dump(mapper, path)
print(f"saved mapper -> {path}")
print("keys:", list(mapper.keys()))
